In [2]:
# ==========================================
# 1. MOUNT GOOGLE DRIVE & EXTRACT ZIP INSTANTLY
# ==========================================
import os
import zipfile
import pandas as pd
import numpy as np
from google.colab import drive

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Since your file is named 'Dataset.zip' in your Drive, we point directly to it
# (Adjust the path if it is placed inside a specific folder, e.g., '/content/drive/MyDrive/YourFolderName/Dataset.zip')
zip_path = '/content/drive/MyDrive/Dataset.zip'

if not os.path.exists(zip_path):
    # Fallback search if it's placed somewhere else in Drive
    print("Dataset.zip not found directly in root /MyDrive. Searching your Drive...")
    found_paths = []
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'Dataset.zip' in files:
            found_paths.append(os.path.join(root, 'Dataset.zip'))

    if found_paths:
        zip_path = found_paths[0]
        print(f"Found your dataset at: {zip_path}")
    else:
        raise FileNotFoundError("Could not find 'Dataset.zip' in your Google Drive. Please check its location.")

extract_dir = '/content/extracted_datasets'
os.makedirs(extract_dir, exist_ok=True)

print("\nExtracting datasets (this takes only a few seconds from Drive)...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print("Extraction complete!")

# Automatically list all extracted CSV files
csv_files = []
for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith('.csv'):
            csv_files.append(os.path.join(root, file))

print("\nFound the following CSV files in your zip:")
for idx, f in enumerate(csv_files):
    print(f"  [{idx}] {f}")

# ==========================================
# 2. FAST LOADING WITH MEMORY OPTIMIZATION
# ==========================================
# Automatically assign paths if standard names match, otherwise prompt safely
train_path = input("\nEnter the full path for your TRAIN CSV from the list above: ").strip()
val_path   = input("Enter the full path for your VALIDATION CSV from the list above: ").strip()
test_path  = input("Enter the full path for your TEST CSV from the list above: ").strip()

target_col = input("Enter your target/label column name: ").strip()

print("\nLoading datasets with memory downcasting (prevents RAM/Runtime crashes)...")

def load_optimized_csv(filepath):
    # Read a small chunk to check data types and downcast to float32/int32
    sample = pd.read_csv(filepath, nrows=1000)
    int_cols = sample.select_dtypes(include=['int64']).columns
    float_cols = sample.select_dtypes(include=['float64']).columns

    dtypes = {col: 'int32' for col in int_cols}
    dtypes.update({col: 'float32' for col in float_cols})

    return pd.read_csv(filepath, dtype=dtypes, low_memory=True)

train_df = load_optimized_csv(train_path)
val_df = load_optimized_csv(val_path)
test_df = load_optimized_csv(test_path)

print(f"\nDatasets loaded successfully!")
print(f"Train shape:      {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape:       {test_df.shape}")

# ==========================================
# 3. EXPLORATORY DATA ANALYSIS (FEATURES & LABELS)
# ==========================================
print("\n" + "="*50)
print("FEATURE OVERVIEW")
print("="*50)
print(f"Total Features (excluding label): {train_df.shape[1] - 1}")

print("\n" + "="*50)
print("TRAIN SET - LABEL DISTRIBUTION & PERCENTAGES")
print("="*50)
train_counts = train_df[target_col].value_counts()
train_percentages = train_df[target_col].value_counts(normalize=True) * 100
train_summary = pd.DataFrame({'Count': train_counts, 'Percentage (%)': train_percentages})
print(train_summary)

print("\n" + "="*50)
print("VALIDATION SET - LABEL DISTRIBUTION & PERCENTAGES")
print("="*50)
val_counts = val_df[target_col].value_counts()
val_percentages = val_df[target_col].value_counts(normalize=True) * 100
val_summary = pd.DataFrame({'Count': val_counts, 'Percentage (%)': val_percentages})
print(val_summary)

# ==========================================
# 4. LIGHTNING-FAST PREPROCESSING (TRAIN & VAL)
# ==========================================
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Separate Features (X) and Target (y)
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_val = val_df.drop(columns=[target_col])
y_val = val_df[target_col]

# Select numerical columns for fast tensor/matrix operations
numerical_cols = X_train.select_dtypes(include=['int32', 'int64', 'float32', 'float64']).columns.tolist()

print(f"\nProcessing {len(numerical_cols)} numerical features for Train and Validation...")

# A. Fast Imputation (Filled using Median computed strictly on Train)
imputer = SimpleImputer(strategy='median')
X_train_num = imputer.fit_transform(X_train[numerical_cols])
X_val_num = imputer.transform(X_val[numerical_cols])

# B. Fast Standardization (Fitted strictly on Train to avoid data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_val_scaled = scaler.transform(X_val_num)

print("\n" + "="*50)
print("PREPROCESSING COMPLETED BLAZINGLY FAST!")
print("="*50)
print(f"Processed X_train shape: {X_train_scaled.shape}")
print(f"Processed X_val shape:   {X_val_scaled.shape}")

Mounting Google Drive...
Mounted at /content/drive

Extracting datasets (this takes only a few seconds from Drive)...
Extraction complete!

Found the following CSV files in your zip:
  [0] /content/extracted_datasets/Dataset/validation.csv
  [1] /content/extracted_datasets/Dataset/test.csv
  [2] /content/extracted_datasets/Dataset/train.csv

Enter the full path for your TRAIN CSV from the list above: /content/extracted_datasets/Dataset/train.csv
Enter the full path for your VALIDATION CSV from the list above:  /content/extracted_datasets/Dataset/validation.csv
Enter the full path for your TEST CSV from the list above: /content/extracted_datasets/Dataset/test.csv
Enter your target/label column name: label

Loading datasets with memory downcasting (prevents RAM/Runtime crashes)...

Datasets loaded successfully!
Train shape:      (5491971, 47)
Validation shape: (1176851, 47)
Test shape:       (1176851, 47)

FEATURE OVERVIEW
Total Features (excluding label): 46

TRAIN SET - LABEL DISTRIBUT

In [3]:
import os
import time
import zipfile
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
import lightgbm as lgb
from google.colab import drive

# ==========================================
# AUTO-MOUNT, AUTO-EXTRACT, AND AUTO-LOAD DATA
# ==========================================
if 'X_train_scaled' not in globals() or 'y_train' not in globals():
    print("🔄 Colab runtime was reset. Restoring dataset state dynamically...")

    # Ensure drive is mounted
    if not os.path.exists('/content/drive'):
        print("Mounting Google Drive...")
        drive.mount('/content/drive')

    zip_path = '/content/drive/MyDrive/Dataset.zip'

    # Fallback search if it's placed somewhere else in Drive
    if not os.path.exists(zip_path):
        print("Dataset.zip not found in root /MyDrive. Searching Google Drive...")
        found_paths = []
        for root, dirs, files in os.walk('/content/drive/MyDrive'):
            if 'Dataset.zip' in files:
                found_paths.append(os.path.join(root, 'Dataset.zip'))
        if found_paths:
            zip_path = found_paths[0]
        else:
            raise FileNotFoundError("Could not find 'Dataset.zip' anywhere in Google Drive.")

    extract_dir = '/content/extracted_datasets'
    os.makedirs(extract_dir, exist_ok=True)

    # Extract dynamically if not already extracted
    train_path = "/content/extracted_datasets/Dataset/train.csv"
    val_path = "/content/extracted_datasets/Dataset/validation.csv"

    if not os.path.exists(train_path) or not os.path.exists(val_path):
        print("Extracting datasets from zip file...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print("Extraction complete!")

    # Find paths dynamically if nested differently
    for root, dirs, files in os.walk(extract_dir):
        for file in files:
            if file.lower() == 'train.csv':
                train_path = os.path.join(root, file)
            elif file.lower() == 'validation.csv':
                val_path = os.path.join(root, file)

    target_col = "label"

    print("Loading optimized train and validation datasets...")
    def load_optimized_csv(filepath):
        sample = pd.read_csv(filepath, nrows=1000)
        int_cols = sample.select_dtypes(include=['int64']).columns
        float_cols = sample.select_dtypes(include=['float64']).columns
        dtypes = {col: 'int32' for col in int_cols}
        dtypes.update({col: 'float32' for col in float_cols})
        return pd.read_csv(filepath, dtype=dtypes, low_memory=True)

    train_df = load_optimized_csv(train_path)
    val_df = load_optimized_csv(val_path)

    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col]
    X_val = val_df.drop(columns=[target_col])
    y_val = val_df[target_col]

    numerical_cols = X_train.select_dtypes(include=['int32', 'int64', 'float32', 'float64']).columns.tolist()

    print("Imputing and scaling features...")
    imputer = SimpleImputer(strategy='median')
    X_train_num = imputer.fit_transform(X_train[numerical_cols])
    X_val_num = imputer.transform(X_val[numerical_cols])

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_num)
    X_val_scaled = scaler.transform(X_val_num)
    print("✅ Datasets prepared! Proceeding to train the models...\n")

# ==========================================
# DEFINE AND RUN THE 4 MODELS
# ==========================================
models = {
    "SGD Logistic Regression": SGDClassifier(loss='log_loss', max_iter=1000, tol=1e-3, random_state=42, n_jobs=-1),
    "SGD Linear SVM": SGDClassifier(loss='hinge', max_iter=1000, tol=1e-3, random_state=42, n_jobs=-1),
    "Decision Tree (Max Depth 10)": DecisionTreeClassifier(max_depth=10, random_state=42),
    "LightGBM Classifier": lgb.LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    )
}

results = {}

print("Starting sequential training of the 4 models...\n")

for name, model in models.items():
    print("="*60)
    print(f"Training Model: {name}")
    print("="*60)

    start_time = time.time()
    model.fit(X_train_scaled, y_train)
    train_time = time.time() - start_time
    print(f"Training finished in {train_time:.2f} seconds.")

    print(f"Evaluating {name} on validation set...")
    y_pred = model.predict(X_val_scaled)

    acc = accuracy_score(y_val, y_pred)
    results[name] = {
        "accuracy": acc,
        "train_time": train_time,
        "model_object": model
    }
    print(f"Validation Accuracy: {acc * 100:.2f}%")
    print(f"Classification Report snippet (first 5 classes shown):")
    print(classification_report(y_val, y_pred, digits=4, zero_division=0, labels=list(set(y_val))[:5]))
    print("\n")

# ==========================================
# SUMMARY OF RESULTS
# ==========================================
print("="*60)
print("SUMMARY OF MODEL PERFORMANCES")
print("="*60)
for name, metrics in results.items():
    print(f"{name:<30} | Accuracy: {metrics['accuracy']*100:.2f}% | Train Time: {metrics['train_time']:.2f}s")

Starting sequential training of the 4 models...

Training Model: SGD Logistic Regression
Training finished in 589.75 seconds.
Evaluating SGD Logistic Regression on validation set...
Validation Accuracy: 79.41%
Classification Report snippet (first 5 classes shown):
                    precision    recall  f1-score   support

    Mirai-udpplain     0.9813    0.9889    0.9851     22314
  DDoS-RSTFINFlood     0.9999    0.9988    0.9994    101563
Mirai-greeth_flood     0.6234    0.8904    0.7333     24910
  Backdoor_Malware     0.0000    0.0000    0.0000        93
 VulnerabilityScan     0.2609    0.0928    0.1369       905

         micro avg     0.9120    0.9732    0.9416    149785
         macro avg     0.5731    0.5942    0.5709    149785
      weighted avg     0.9294    0.9732    0.9472    149785



Training Model: SGD Linear SVM
Training finished in 352.62 seconds.
Evaluating SGD Linear SVM on validation set...
Validation Accuracy: 79.09%
Classification Report snippet (first 5 classes 

Exception ignored on calling ctypes callback function <function _log_callback at 0x7ee6139d9440>:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
KeyboardInterrupt: 


No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

Exception ignored on calling ctypes callback function <function _log_callback at 0x7ee6139d9440>:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
KeyboardInterrupt: 


No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

...

No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


Exception ignored on calling ctypes callback function <function _log_callback at 0x7ee6139d9440>:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
KeyboardInterrupt: 


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Validation Accuracy: 29.85%
Classification Report snippet (first 5 classes shown):
                    precision    recall  f1-score   support

    Mirai-udpplain     0.4276    0.9224    0.5844     22314
  DDoS-RSTFINFlood     0.0000    0.0000    0.0000    101563
Mirai-greeth_flood     0.0704    0.6905    0.1277     24910
  Backdoor_Malware     0.0000    0.0000    0.0000        93
 VulnerabilityScan     0.0000    0.0000    0.0000       905

         micro avg     0.1235    0.2523    0.1658    149785
         macro avg     0.0996    0.3226    0.1424    149785
      weighted avg     0.0754    0.2523    0.1083    149785



SUMMARY OF MODEL PERFORMANCES
SGD Logistic Regression        | Accuracy: 79.41% | Train Time: 589.75s
SGD Linear SVM                 | Accuracy: 79.09% | Train Time: 352.62s
Decision Tree (Max Depth 10)   | Accuracy: 95.28% | Train Time: 83.17s
LightGBM Classifier            | Accuracy: 29.85% | Train Time: 1347.68s
